In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# parameters for hard sphere interactions
omega = 0.5
# corresponds to isotropic scattering
alpha = 1
mu_inf_mu_1 = 1.016034
mu_1_mu_inf =  1 / mu_inf_mu_1
mu_ref = 2.117e-5

m = 66.3e-27

k_B = 1.380658e-23
T_init = 273.15
numerator =  5 * (alpha + 1) * (alpha + 2) * np.sqrt(m * k_B * T_init / np.pi)
denominator = 4 * alpha * (5 - 2 * omega) * (7 - 2 * omega) * mu_ref * (mu_1_mu_inf)
d_ref =  np.sqrt(numerator / denominator)
print(f"Reference Diameter : {d_ref:0.4E}")
pressure = 266.644
N_A = 6.022140e23
R = 8.314472
density = pressure * N_A / (R * T_init)
lambda_0 =  1 / (np.sqrt(2) * np.pi * d_ref**2 * density)
print(f"Lambda_0           : {lambda_0:0.4E}")

L = 1e-3
cell_counts =
dx = L / cell_counts
dx_tilde = dx / lambda_0
print(f"Delta x tilde      : {dx_tilde:0.4E}")

c_0 = np.sqrt(2 * k_B * T_init / m)
print(f"c_0                : {c_0:0.4E}")
dt = 5e-8
t_0 = lambda_0 / c_0

dt_tilde = dt / t_0
print(f"Delta t tilde      : {dt_tilde:0.4E}")

Reference Diameter : 3.6579E-10
Lambda_0           : 2.3792E-05
Delta x tilde      : 1.0007E+00
c_0                : 3.3729E+02
Delta t tilde      : 7.0884E-01


In [ ]:
def q_wall(dt_tilde, dx_tilde):
  factor = 1 + 0.0281 * dt_tilde**2
  factor += 0.0399 * dx_tilde**2
  factor -= 0.0011 * dx_tilde**4
  factor -= 0.0170 * dt_tilde**2 * dx_tilde**2
  factor += 0.0083 * dt_tilde**4 * dx_tilde**2
  return 1512 * factor

expected_q_wall = q_wall(dt_tilde, dx_tilde)
print(f"Expected wall heat flux : {expected_q_wall:0.4E}")

Expected wall heat flux : 1.5356E+03


In [4]:
def trueSolution(x, T_mid, delta_T):
  T_min = T_mid - delta_T / 2.0
  return T_min + (delta_T) / (x.max() - x.min()) * x

SyntaxError: expected ':' (3544287116.py, line 1)

In [ ]:

average = None
x_vals = None
runs = 15

T_mid = 273.15
delta_T = 100
show_all = True
all_temps = None
fails = 0
left_fluxes = np.zeros((runs))
right_fluxes = np.zeros((runs))

plt.figure()
for i in range(runs):
  if i == 12 or i == 28:
    fails += 1
    continue

  data = pd.read_csv(f"30_ppe/run_{i:d}_average_temperature_4001.csv")
  flux_data = pd.read_csv(f"30_ppe/run_{i:d}.csv")
  if np.isnan(flux_data["left_heat_flux"].to_numpy()[0]) or np.isnan(flux_data["right_heat_flux"].to_numpy()[0]):
    fails += 1
    continue
  left_fluxes[i - fails] = flux_data["left_heat_flux"].to_numpy()[0]
  right_fluxes[i - fails] = flux_data["right_heat_flux"].to_numpy()[0]
  print(flux_data)
  temperature = data["time_averaged_temperature"].to_numpy()
  if i == 0:
    all_temps = np.zeros((runs, len(temperature)))
    x_vals = data["x"].to_numpy() * 1e3
  all_temps[i - fails, :] = temperature
  if show_all:
    plt.plot(x_vals, data["time_averaged_temperature"], label=f"Run {i}", alpha=0.25)

T_average = all_temps[:-fails].mean(axis=0)
T_std = all_temps[:-fails].std(axis=0)

print(f"{fails:d} Simulations Failed")
plt.plot(x_vals, trueSolution(x_vals, T_mid, delta_T), 'k--', label="Continuum" )
plt.plot(x_vals, T_average, label='Simulated')
plt.fill_between(x_vals, T_average - 2 * T_std, T_average + 2 * T_std, color='tab:blue', alpha=0.25)
#plt.ylim(bottom=T_mid - delta_T / 2, top =T_mid + delta_T / 2)
plt.xlim(left=0, right=1)
plt.xlabel("x (mm)")
plt.ylabel("Temperature (K)")
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '30_ppe/run_0_average_temperature_4001.csv'

<Figure size 640x480 with 0 Axes>

In [ ]:
from random import seed, randint
seed(0)
print(randint(0, int(1e8)))
print(randint(0, int(1e8)))

51706749
56448162


In [ ]:
print(f"Left Flux {left_fluxes[:-fails].mean()}")
print(f"Right Flux {right_fluxes[:-fails].mean()}")

Left Flux 1610.6690465226848
Right Flux -1602.4192660139001
